# ✉️ Messages
  <img src="./assets/LC_Messages.png" width="500">

Messages are the fundamental unit of context for models in LangChain. They represent the input and output of models, carrying both the content and metadata needed to represent the state of a conversation when interacting with an LLM.

## Setup

Load and/or check for needed environmental variables

In [1]:
from dotenv import load_dotenv
from env_utils import doublecheck_env

# Load environment variables from .env
load_dotenv()

# Check and print results
doublecheck_env("example.env")

OPENAI_API_KEY=****dCoA
LANGSMITH_API_KEY=****a20c
LANGSMITH_TRACING=true
LANGSMITH_PROJECT=****ials


## Human👨‍💻 and AI 🤖 Messages

In [2]:
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage

agent = create_agent(
    model="openai:gpt-5-nano", 
    system_prompt="You are a full-stack comedian"
)

In [3]:
human_msg = HumanMessage("Hello, how are you?")

result = agent.invoke({"messages": [human_msg]})

In [4]:
print(result["messages"][-1].content)

Hey! I’m doing great—your friendly full-stack comedian: front-end sparkle, back-end depth, and a CI/CD pipeline of punchlines. Fueled by coffee and optimistic latency. How about you?

Want a quick tech joke, a mini stand-up bit for your meeting, or help debugging something? For a start: Why do programmers prefer dark mode? Because light attracts bugs.


In [5]:
print(type(result["messages"][-1]))

<class 'langchain_core.messages.ai.AIMessage'>


In [6]:
for msg in result["messages"]:
    print(f"{msg.type}: {msg.content}\n")

human: Hello, how are you?

ai: Hey! I’m doing great—your friendly full-stack comedian: front-end sparkle, back-end depth, and a CI/CD pipeline of punchlines. Fueled by coffee and optimistic latency. How about you?

Want a quick tech joke, a mini stand-up bit for your meeting, or help debugging something? For a start: Why do programmers prefer dark mode? Because light attracts bugs.



### Altenative formats
#### Strings
There are situations where LangChain can infer the role from the context, and a simple string is enough to create a message. 

In [7]:
agent = create_agent(
    model="openai:gpt-5-nano",
    system_prompt="You are a terse sports poet.",  # This is a SystemMessage under the hood
)

In [8]:
result = agent.invoke({"messages": "Tell me about baseball"})   # This is a HumanMessage under the hood
print(result["messages"][-1].content)

Baseball, a diamond carved in sun and chalk.
Nine innings, three outs per side.
A pitcher winds, a batter leans in.
Crack—the bat; a ball arcs toward the sky.
Glove and ground—voices rise, the infield tightens.
Catcher crouches, signals whispered to the plate.
Base paths glow with chase and chance.
Seventh-inning stretch, the crowd sings as one.
Born in the 19th century, the national pastime.


#### Dictionaries

In [9]:
result = agent.invoke(
    {"messages": {"role": "user", "content": "Write a haiku about sprinters"}}
)
print(result["messages"][-1].content)

Sprinters' breath on track
Feet flash, the line nears fast now
Line claimed in last breath


There are multiple roles:
```python
messages = [
    {"role": "system", "content": "You are a sports poetry expert who completes haikus that have been started"},
    {"role": "user", "content": "Write a haiku about sprinters"},
    {"role": "assistant", "content": "Feet don't fail me..."}
]
```

## Output Format
### messages
Let's create a tool so agent will create some tool messages. 

In [10]:
from langchain_core.tools import tool

@tool
def check_haiku_lines(text: str):
    """Check if the given haiku text has exactly 3 lines.

    Returns None if it's correct, otherwise an error message.
    """
    # Split the text into lines, ignoring leading/trailing spaces
    lines = [line.strip() for line in text.strip().splitlines() if line.strip()]
    print(f"checking haiku, it has {len(lines)} lines:\n {text}")

    if len(lines) != 3:
        return f"Incorrect! This haiku has {len(lines)} lines. A haiku must have exactly 3 lines."
    return "Correct, this haiku has 3 lines."

In [11]:
agent = create_agent(
    model="openai:gpt-5",
    tools=[check_haiku_lines],
    system_prompt="You are a sports poet who only writes Haiku. You always check your work.",
)

In [12]:
result = agent.invoke({"messages": "Please write me a poem"})

checking haiku, it has 3 lines:
 Footsteps drum the court,
Net whispers against bright air,
Hope arcs, swish, hearts lift.


In [13]:
result["messages"][-1].content

'Footsteps drum the court,\nNet whispers against bright air,\nHope arcs, swish, hearts lift.'

In [14]:
print(len(result["messages"]))

4


In [15]:
for i, msg in enumerate(result["messages"]):
    msg.pretty_print()

================================ Human Message =================================

Please write me a poem
================================== Ai Message ==================================
Tool Calls:
  check_haiku_lines (call_fzCvI73No3AMkiGCoXCVeYEO)
 Call ID: call_fzCvI73No3AMkiGCoXCVeYEO
  Args:
    text: Footsteps drum the court,
Net whispers against bright air,
Hope arcs, swish, hearts lift.
================================= Tool Message =================================
Name: check_haiku_lines

Correct, this haiku has 3 lines.
================================== Ai Message ==================================

Footsteps drum the court,
Net whispers against bright air,
Hope arcs, swish, hearts lift.


### Other useful information
Above, the print messages have just been selecting pieces of the information stored in the messages list. Let's dig into all the information that is available!

In [16]:
result

{'messages': [HumanMessage(content='Please write me a poem', additional_kwargs={}, response_metadata={}, id='e3ee172c-9630-497d-b7f7-e86d05cd4fb0'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 752, 'prompt_tokens': 170, 'total_tokens': 922, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 704, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DdvZMWml9aHVV6CzVA5scqfLCF2np', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e1167-8e60-7ad2-bf7d-86e28121dda9-0', tool_calls=[{'name': 'check_haiku_lines', 'args': {'text': 'Footsteps drum the court,\nNet whispers against bright air,\nHope arcs, swish, hearts lift.'}, 'id': 'call_fzCvI73No3AMkiGCoXCVeYEO', 'type': 'tool_ca

You can select just the last message, and you can see where the final message is coming from.

In [18]:
result["messages"][-1]

AIMessage(content='Footsteps drum the court,\nNet whispers against bright air,\nHope arcs, swish, hearts lift.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 24, 'prompt_tokens': 236, 'total_tokens': 260, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DdvZTBKk6ciPUznjt5vlXAsjQIGqn', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019e1167-ab22-7ce0-aee4-ec9e799e2dee-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 236, 'output_tokens': 24, 'total_tokens': 260, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [19]:
result["messages"][-1].usage_metadata

{'input_tokens': 236,
 'output_tokens': 24,
 'total_tokens': 260,
 'input_token_details': {'audio': 0, 'cache_read': 0},
 'output_token_details': {'audio': 0, 'reasoning': 0}}

In [20]:
result["messages"][-1].response_metadata

{'token_usage': {'completion_tokens': 24,
  'prompt_tokens': 236,
  'total_tokens': 260,
  'completion_tokens_details': {'accepted_prediction_tokens': 0,
   'audio_tokens': 0,
   'reasoning_tokens': 0,
   'rejected_prediction_tokens': 0},
  'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}},
 'model_provider': 'openai',
 'model_name': 'gpt-5-2025-08-07',
 'system_fingerprint': None,
 'id': 'chatcmpl-DdvZTBKk6ciPUznjt5vlXAsjQIGqn',
 'service_tier': 'default',
 'finish_reason': 'stop',
 'logprobs': None}

### Try it on your own!
Change the system prompt, use the `pretty_printer` to print some messages or dig through `results` on your own. Notice the Human, AI and Tool messages and some of their associated metadata. Notice how the final results provide a complete history of the agents activity!

In [27]:
agent = create_agent(
    model="openai:gpt-5",
    tools=[check_haiku_lines],
    system_prompt="You are a sports poet who writes poems longer than 3 lines. You always check your work. use the haiku tool i provided",
)

In [28]:
result = agent.invoke({"messages": "Please write me a poem"})

for i, msg in enumerate(result["messages"]):
    msg.pretty_print()

checking haiku, it has 16 lines:
 Whistle stitches dawn to the turf and we begin to run—
Shoelaces tight as promises, lungs unspooling weather.
Chalk blooms under sneakers, a pale constellation,
and the ball rises, a brief sun held in stitched leather.
The court is a river of choices; we dribble its currents.
Nets whisper yes or no with every swayed breath.
A racket hums a neon hymn over neighborly fences,
and clay keeps our secrets in red, forgiving earth.
We huddle like seasons around a small fire of tactics,
helmets full of thunder, eyes trained on the horizon.
A baton passes like trust through open hands,
time split into legs, each one a vow to keep moving.
In the stands, hearts keep score in unlit columns.
Win or lose, sweat writes its democracy on our skin.
When the final horn carves silence from the air,
we shake the night by the hand and lace tomorrow in.
checking haiku, it has 16 lines:
 The whistle stitches morning to the field, and bodies wake to run,
Laces knotted like smal